# Infer recombinant PPE18 sequences for each GC Event detected

Useful links:
https://help.iedb.org/hc/en-us/articles/114094152371-What-thresholds-cut-offs-should-I-use-for-MHC-class-I-and-II-binding-predictions
https://nextgen-tools.iedb.org/docs/tools/tcell_ii/index.html



# Import Statements (libraries + Functions)

In [122]:
import numpy as np
import pandas as pd
import vcf
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

%matplotlib inline

In [123]:
#import sklearn

In [124]:
import scipy.stats
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import spearmanr
from scipy.stats import ranksums

In [125]:
#from Bio import SeqIO

In [126]:
#import io

In [127]:
import json

In [128]:
# import screed
# import mmh3

In [129]:
#import subprocess

In [130]:
#from pycirclize import Circos

In [131]:
from matplotlib.patches import Patch
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
import matplotlib.ticker as mticker


In [132]:
# Define matplotlib plot style from config file
plt.style.use('../mgm.v1.mplstyle')

In [133]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [134]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf
#import bioframe.vis

#### Define sets of coord cols for using BioFrame

In [135]:
Query_CoordCols = ("Query_Name", "Query_Start", "Query_End")
HmReg_CoordCols = ("Chr", "Start", "End")
HmRegion_CoordCols = HmReg_CoordCols
Epitope_CoordCols = ("Chrom", "Rv_Start", "Rv_End")
RE_CoordCols = ("seqname", "start_0based", "end_1based")
GenomeAnno_CoordCols = ("Chrom", "Start", "End")


#### Pandas Viewing Settings

In [136]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Defining Functions

## Define functions for adding AA substitions to protein sequence 

In [137]:
from typing import Union, List, Tuple
import pandas as pd
from Bio.Seq import Seq
import pandas as pd

def apply_aa_substitutions(
    protein_seq: Union[Seq, str],
    subs_df: pd.DataFrame,
    ref_col: str = "Ref_AA",
    mut_col: str = "Mut_AA",
    pos_col: str = "Codon_0based",
    strict: bool = False,
) -> Tuple[Union[Seq, str], List[Tuple[int, str, str, str]]]:
    """
    Apply amino-acid substitutions to a protein sequence using iterrows().

    Parameters
    ----------
    protein_seq : Seq or str
        Original protein sequence.
    subs_df : pd.DataFrame
        Must contain [Codon_0based, Ref_AA, Mut_AA].
    ref_col, mut_col, pos_col : str
        Column names in the DataFrame.
    strict : bool
        If True, raise ValueError on any reference-AA mismatch.

    Returns
    -------
    new_seq : Seq or str
        Sequence with substitutions applied (same type as input).
    mismatches : list of tuples
        [(pos, seq_ref, df_ref, df_mut), ...] for any ref-AA mismatches.
    """
    if pos_col not in subs_df.columns:
        raise KeyError(f"Required column '{pos_col}' not found in substitutions DataFrame.")

    input_was_seq = isinstance(protein_seq, Seq)
    seq_list = list(str(protein_seq))
    mismatches: List[Tuple[int, str, str, str]] = []

    for _, row in subs_df.iterrows():
        pos = int(row[pos_col])  # already 0-based
        ref = str(row[ref_col]).upper()
        mut = str(row[mut_col]).upper()

        if pos < 0 or pos >= len(seq_list):
            raise IndexError(f"Position {pos} out of range for sequence length {len(seq_list)}.")

        seq_ref = seq_list[pos].upper()
        if seq_ref != ref:
            if strict:
                raise ValueError(
                    f"Ref mismatch at pos {pos}: sequence has '{seq_ref}', dataframe has '{ref}'"
                )
            mismatches.append((pos, seq_ref, ref, mut))

        seq_list[pos] = mut
        print(f"Mutated position {pos}: {seq_ref} → {mut}")

    new_seq = "".join(seq_list)
    return (Seq(new_seq) if input_was_seq else new_seq), mismatches


def make_unq_aa_subs_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a dataframe of unique amino acid substitutions with 0-based codon index.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with at least ["Codon", "Ref_AA", "Mut_AA"].

    Returns
    -------
    pd.DataFrame
        Dataframe with unique rows and an extra column "Codon_0based".
    """
    if not all(col in df.columns for col in ["Codon", "Ref_AA", "Mut_AA"]):
        raise KeyError("Input dataframe must have columns ['Codon', 'Ref_AA', 'Mut_AA'].")

    tmp = df[["Codon", "Ref_AA", "Mut_AA"]].drop_duplicates().reset_index(drop=True)
    tmp["Codon_0based"] = tmp["Codon"] - 1
    return tmp


from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

def write_AAseq_to_fasta(seq: Seq, header: str, out_fasta: str) -> None:
    """
    Write a Seq object to a FASTA file.

    Parameters
    ----------
    seq : Seq
        Amino acid sequence as a Biopython Seq object.
    header : str
        FASTA header (sequence identifier).
    out_fasta : str
        Path to output FASTA file.
    """
    record = SeqRecord(seq, id=header, description="")
    SeqIO.write(record, out_fasta, "fasta")

### Define `DNA-Features-Viewer` Preprocessing Functions

In [138]:

def generate_Event_GraphicFeatures(i_GC_Events_DF):

    L_GFeats = []
    
    for i, row in i_GC_Events_DF.iterrows():
        
        Rv_Start = row["start_0based"]
        Rv_End = row["end_1based"]
        i_Event_ID = row["EventID"]

        Event_Feat = GraphicFeature(start = Rv_Start , end = Rv_End,
                                      label = i_Event_ID[-3:],
                                      linewidth = 0.5,
                                      color = "#CBC3E3", #"purple",
                                      thickness = 5, 
                                      fontdict = {"fontsize": 4},
                                      linecolor = "black")
        
        L_GFeats.append(Event_Feat)

    return L_GFeats


def AddEvents_ToGraphicRecord(Graphic_Record, i_GC_Events_DF):

    L_Event_GFeats = generate_Event_GraphicFeatures(i_GC_Events_DF)
    
    Graphic_Record.features = Graphic_Record.features + L_Event_GFeats

    return Graphic_Record


def generate_Epitope_GraphicFeatures(i_Epitopes_DF):

    L_GFeats = []
    
    for i, row in i_Epitopes_DF.iterrows():
        
        Rv_Start = row["Rv_Start"]
        Rv_End = row["Rv_End"]
        i_epitope_seq = row["Epitope_Seq"]
        i_epitope_ID = row["Epitope_ID"]

        Epitope_Feat = GraphicFeature(start = Rv_Start , end = Rv_End ,
                                      #label = i_epitope_ID,
                                      color = "darkred",
                                      linecolor = "black")
        
        L_GFeats.append(Epitope_Feat)

    return L_GFeats


def generate_AssayedEpitope_GraphicFeatures(i_Epitopes_DF):

    L_GFeats = []
    
    for i, row in i_Epitopes_DF.iterrows():
        
        Rv_Start = row["Rv_Start"]
        Rv_End = row["Rv_End"]
        i_epitope_seq = row["Epitope_Seq"]
        i_epitope_ID = row["Epitope_ID"]
        i_PosEpitope = row["PosEpitope_Any"]
        if i_PosEpitope == True:
    
            Epitope_Feat = GraphicFeature(start = Rv_Start , end = Rv_End ,
                                          #label = i_epitope_ID,
                                          linewidth = 0.5,
                                          color = "darkred",
                                          thickness = 5,
                                          linecolor = "black")
            
        else:
            Epitope_Feat = GraphicFeature(start = Rv_Start , end = Rv_End ,
                                          #label = i_epitope_ID,
                                          #linewidth = 0.5,
                                          color = "lightgrey", alpha = 0.3,
                                          thickness = 5,
                                          linewidth = 0.5,
                                          linecolor = "black")

            
        L_GFeats.append(Epitope_Feat)

    return L_GFeats


def AddEpitopes_ToGraphicRecord(Graphic_Record, i_Epitopes_DF):

    L_Epitope_GFeats = generate_Epitope_GraphicFeatures(i_Epitopes_DF)
    
    Graphic_Record.features = Graphic_Record.features + L_Epitope_GFeats

    return Graphic_Record




## Functions for Composite vizs of results + gene annotations
`H37Rv Gene Annoations` + `Epitope Mapping`, + `mutation frequency` + `GC Events`, `etc`

In [139]:
from typing import Optional, Tuple, Dict
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from typing import Optional, Tuple, Dict
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

def plot_Anno_Epitope_MutFreq(
    Viz_Start: int,
    Viz_End: int,
    in_PerCodon_MutCt_DF,                 # cols: Pos_0based, Mutation_Count
    in_Genome_Graphic_Record,             # dna_features_viewer GraphicRecord (H37Rv)
    in_AllPeptides_GraphicRecord,         # GraphicRecord of all assayed peptides
    in_EpitopeCov_DF,                     # cols: start, end, Cov_PosEpitopes
    *,
    mut_ylim: Optional[Tuple[float, float]] = (0, 15),
    figsize: Tuple[float, float] = (7, 3),
    height_ratios: Tuple[int, int, int, int] = (3, 5, 1, 4),
    cmap_name: str = "Reds",
    point_size: float = 2.0,
    vline_lw: float = 0.9,
    epi_linewidth: float = 2.5,
    genome_label_threshold: int = 5,
    show_bottom_xticks: bool = True,
    tight_layout: bool = True,
    epi_global_max: Optional[float] = None,   # 👈 new option
) -> Tuple[plt.Figure, Dict[str, plt.Axes]]:
    """
    Plot genomic annotations, per-codon mutation counts, epitope coverage summary,
    and assayed peptides for a window [Viz_Start, Viz_End].

    Epitope coverage colors are normalized to the global max of Cov_PosEpitopes
    across the entire in_EpitopeCov_DF.
    """
    # --- Crop graphic records to region ---
    Graphic_Record_cropped = in_Genome_Graphic_Record.crop((Viz_Start, Viz_End + 1))
    AllPeptides_Records_cropped = in_AllPeptides_GraphicRecord.crop((Viz_Start, Viz_End + 1))

    # --- Create figure & axes ---
    fig, axs = plt.subplots(
        4, 1, figsize=figsize,
        gridspec_kw={'height_ratios': height_ratios}
    )
    Genome_Anno_ax, MF_ax, EpiSumm_ax, AllAssayedPeptides_ax = axs

    # --- Gene Annotations ---
    if hasattr(Graphic_Record_cropped, "plot"):
        Graphic_Record_cropped.plot(strand_in_label_threshold=genome_label_threshold,
                                    ax=Genome_Anno_ax)
    Genome_Anno_ax.set_xlim(Viz_Start, Viz_End)
    Genome_Anno_ax.set_xticks([])

    # --- Mutation Counts ---
    if {"Pos_0based", "Mutation_Count"}.issubset(in_PerCodon_MutCt_DF.columns):
        mut_df = in_PerCodon_MutCt_DF.query(
            f"(Pos_0based + 1) >= {Viz_Start} & (Pos_0based + 1) <= {Viz_End}"
        ).copy()
        if not mut_df.empty:
            xvals = mut_df["Pos_0based"].to_numpy() + 1
            yvals = mut_df["Mutation_Count"].to_numpy()
            MF_ax.vlines(x=xvals, ymin=0, ymax=yvals, color="black", linewidth=vline_lw, alpha=0.6)
            MF_ax.scatter(x=xvals, y=yvals, color="red", s=point_size, alpha=1)

    MF_ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    MF_ax.ticklabel_format(style='plain', axis='x')
    MF_ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    MF_ax.set_xlim(Viz_Start, Viz_End)
    if mut_ylim is not None:
        MF_ax.set_ylim(*mut_ylim)
    MF_ax.set_xticks([])
    sns.despine(ax=MF_ax)

    # --- Epitope density summary ---
    cmap = plt.get_cmap(cmap_name)
    if {"start", "end", "Cov_PosEpitopes"}.issubset(in_EpitopeCov_DF.columns):
        Reg_EpiCov_DF = in_EpitopeCov_DF.query(
            f"start >= {Viz_Start} & end <= {Viz_End}"
        ).copy()
        if not Reg_EpiCov_DF.empty:
            if epi_global_max is None:
                global_max = in_EpitopeCov_DF["Cov_PosEpitopes"].max()
            else:
                global_max = epi_global_max
                
            norm = plt.Normalize(vmin=0, vmax=global_max if global_max > 0 else 1)
            for _, row in Reg_EpiCov_DF.iterrows():
                cov = row["Cov_PosEpitopes"]
                if cov > 0:
                    EpiSumm_ax.vlines(x=row["start"], ymin=0, ymax=1,
                                      color=cmap(norm(cov)), linewidth=epi_linewidth, alpha=1)

    EpiSumm_ax.set_xlim(Viz_Start, Viz_End)
    EpiSumm_ax.set_ylim(0, 1)
    EpiSumm_ax.set_xticks([])
    EpiSumm_ax.set_yticks([])
    for spine in ("left", "top", "right"):
        EpiSumm_ax.spines[spine].set_visible(False)
    EpiSumm_ax.set_xlabel("")

    # --- Assayed peptides ---
    #if hasattr(AllPeptides_Records_cropped, "plot"):
    AllPeptides_Records_cropped.plot(strand_in_label_threshold=genome_label_threshold,
                                         ax=AllAssayedPeptides_ax)
    
    AllAssayedPeptides_ax.set_xlim(Viz_Start, Viz_End)
    AllAssayedPeptides_ax.set_ylim(-0.5, 5.5)


    # full integers on x-axis (no sci/offset)
    formatter = mticker.ScalarFormatter(useOffset=False)
    formatter.set_scientific(False)
    AllAssayedPeptides_ax.xaxis.set_major_formatter(formatter)
    AllAssayedPeptides_ax.ticklabel_format(style='plain', axis='x', useOffset=False)
    AllAssayedPeptides_ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=6, integer=True))
    
    if tight_layout:
        fig.tight_layout()

    axes = {
        "Genome_Anno_ax": Genome_Anno_ax,
        "MF_ax": MF_ax,
        "EpiSumm_ax": EpiSumm_ax,
        "AllAssayedPeptides_ax": AllAssayedPeptides_ax,
    }
    return fig, axes


### Functions - composite viz of `epitopes`,`mutation-freq`, and `GC events`

In [140]:
# Define all functions from scratch, then apply them to esxK/L and PPE18 regions.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
from pathlib import Path


# ------------------------
# Constants (colors + sizes)
# ------------------------
NT_COLORS = {"A": "green", "T": "red", "C": "blue", "G": "orange"}
EVENT_BAR_COLOR = "#C9A0DC"   # light purple

# ------------------------
# Helper functions
# ------------------------
def _to_int_series(s: pd.Series) -> pd.Series:
    """Convert a Series to Int64 safely (NaNs preserved)."""
    return pd.to_numeric(s, errors="coerce").astype("Int64")

def select_events_overlapping_region(pGCE_df: pd.DataFrame, start1: int, end1: int) -> pd.DataFrame:
    """
    Return events whose intervals intersect [start1, end1] (1-based, inclusive).
    Expects event columns: 'start_0based' and 'end_1based'.
    Adds '__start1__' and '__end1__' (both floats, 1-based) to the result.
    """
    start0 = _to_int_series(pGCE_df["start_0based"])
    end1b  = _to_int_series(pGCE_df["end_1based"])
    s1 = start0 + 1           # convert to 1-based
    e1 = end1b                # already 1-based
    mask = (e1 >= start1) & (s1 <= end1)
    out = pGCE_df.loc[mask].copy()
    out["__start1__"] = s1[mask].astype(float)
    out["__end1__"]   = e1[mask].astype(float)
    return out

def prepare_snps_positions(snps_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a '__pos1__' column (1-based position) using Pos_0based or Pos_1based.
    Keeps all other columns intact.
    """
    out = snps_df.copy()
    if "Pos_0based" in out.columns:
        out["__pos1__"] = out["Pos_0based"].astype(int) + 1
    elif "Pos_1based" in out.columns:
        out["__pos1__"] = out["Pos_1based"].astype(int)
    else:
        raise ValueError("SNPs DF must include 'Pos_0based' or 'Pos_1based'.")
    return out

def _format_x_plain(ax):
    """Force full integer tick labels (no scientific notation, no offset)."""
    formatter = mticker.ScalarFormatter(useOffset=False)
    formatter.set_scientific(False)
    ax.xaxis.set_major_formatter(formatter)
    ax.ticklabel_format(style='plain', axis='x', useOffset=False)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=6, integer=True))

def _draw_event_rect(ax, y_center: float, x0: float, x1: float, bar_height: float, color: str = EVENT_BAR_COLOR):
    """Draw an event as a filled rectangle centered at y_center with given height."""
    width = max(0.0, x1 - x0)
    rect = Rectangle((x0, y_center - bar_height/2), width, bar_height,
                     facecolor=color, edgecolor='none', alpha=0.9)
    ax.add_patch(rect)

# ------------------------
# Plotting functions
# ------------------------
def plot_multi_event_snp_stack(
    region_start1: int,
    region_end1:   int,
    pGCE_df: pd.DataFrame,
    snps_df: pd.DataFrame,
    *,
    bar_height: float = 0.5,         # thickness of each event bar
    snp_line_width: float = 0.9, #1.2,     # SNP tick thickness
    label_fontsize: int = 8,
    fig_width: float = 9.0,
    dpi: int = 180,
    title: str | None = None,
    outfile: str | None = None,
):
    """
    Stacked view: one row per overlapping event. Labels are placed slightly ABOVE each event.
    SNP ticks span the same height as the event bar.
    """
    # Choose events & SNPs in the window
    events = select_events_overlapping_region(pGCE_df, region_start1, region_end1)
    if events.empty:
        raise ValueError("No events overlap the requested region.")
    events = events.sort_values(["__start1__", "__end1__"]).reset_index(drop=True)

    snps = prepare_snps_positions(snps_df)
    snps_in_region = snps[(snps["__pos1__"] >= region_start1) & (snps["__pos1__"] <= region_end1)].copy()

    # Layout
    n = len(events)
    fig_height = max(1.5, 0.38 * n + 1.0)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=dpi)
    y_levels = np.arange(n, dtype=float)[::-1]  # top to bottom

    # Draw
    for idx, (_, ev) in enumerate(events.iterrows()):
        y = y_levels[idx]
        x0 = max(region_start1, float(ev["__start1__"]))
        x1 = min(region_end1,   float(ev["__end1__"]))
        _draw_event_rect(ax, y, x0, x1, bar_height, EVENT_BAR_COLOR)

        # SNPs (same height)
        s_ev = snps_in_region[snps_in_region["EventID"] == ev["EventID"]]
        if not s_ev.empty:
            ymin = y - bar_height/2
            ymax = y + bar_height/2
            for _, r in s_ev.iterrows():
                xv = float(r["__pos1__"])
                nt = str(r.get("Child_Call", "")).upper()[:1]
                color = NT_COLORS.get(nt, "black")
                ax.vlines(x=xv, ymin=ymin, ymax=ymax, lw=snp_line_width, color=color, alpha=0.95)

        # Label above the bar
        x_mid = (x0 + x1) / 2.0
        ax.text(x_mid, y + 0.7 * bar_height, str(ev["EventID"]),
                ha="center", va="bottom", fontsize=label_fontsize)

    # Cosmetics
    ax.set_xlim(region_start1, region_end1)
    ax.set_ylim(-0.8, len(y_levels) - 1 + 0.8)
    ax.set_yticks([])
    ax.set_xlabel("Genomic position (1-based)")
    _format_x_plain(ax)
    ax.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.4)

    from matplotlib.lines import Line2D
    legend_elems = [Line2D([0], [0], color=NT_COLORS[k], lw=2, label=k) for k in ["A","T","C","G"] if k in NT_COLORS]
    ax.legend(handles=legend_elems, title="Mut allele", fontsize=8, title_fontsize=9, loc="upper right")

    if title is None:
        title = f"Events & SNPs in region {region_start1}-{region_end1}"
    ax.set_title(title, fontsize=10)

    fig.tight_layout()
    if outfile:
        fig.savefig(outfile, bbox_inches="tight")
        return fig, ax, outfile
    return fig, ax, None
import numpy as np
import pandas as pd
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

def plot_multi_event_snp_packed_on_ax(
    ax,
    region_start1: int,
    region_end1:   int,
    pGCE_df: pd.DataFrame,
    snps_df: pd.DataFrame,
    *,
    min_gap: int = 0,                 # bp gap required to share a row
    bar_height: float = 0.5,          # thickness of each event bar
    snp_line_width: float = 1.2,      # SNP tick thickness
    label_fontsize: int = 8,
    nt_colors: dict | None = None,    # e.g. {"A":"green","T":"red","C":"blue","G":"orange"}
    event_bar_color: str = "#C9A0DC", # light purple
    hide_spines: tuple[str, ...] = ("left", "top", "right"),
    show_xlabel: bool = True,
    title: str | None = None,
    add_legend: bool = False,
):
    """
    Packed view: non-overlapping events (within [region_start1, region_end1], 1-based)
    share rows to save vertical space. Draws onto the provided Matplotlib Axes `ax`.

    Returns
    -------
    ax, summary
      summary = {"nrows": int, "events_plotted": list[str], "window": (start1, end1)}
    """
    # defaults
    if nt_colors is None:
        nt_colors = {"A":"green", "T":"red", "C":"blue", "G":"orange"}

    # --- Select events that overlap the window (convert to 1-based) ---
    s1 = pd.to_numeric(pGCE_df["start_0based"], errors="coerce").astype("Int64") + 1
    e1 = pd.to_numeric(pGCE_df["end_1based"],   errors="coerce").astype("Int64")
    mask = (e1 >= region_start1) & (s1 <= region_end1)
    events = pGCE_df.loc[mask].copy()
    if events.empty:
        print("No events overlap the requested region.")
        return ax, None
        #raise ValueError("No events overlap the requested region.")
        
    events["__start1__"] = s1[mask].astype(float)
    events["__end1__"]   = e1[mask].astype(float)

    # Clip to window & sort for packing
    events["__clip_start__"] = np.maximum(events["__start1__"].values, region_start1)
    events["__clip_end__"]   = np.minimum(events["__end1__"].values,   region_end1)
    events = events.sort_values(["__clip_start__", "__clip_end__"]).reset_index(drop=True)

    # --- Greedy row packing (no overlaps on the same row) ---
    rows_last_end = []
    row_of_event = []
    for _, ev in events.iterrows():
        placed = False
        s = float(ev["__clip_start__"])
        e = float(ev["__clip_end__"])
        for ri in range(len(rows_last_end)):
            if s > (rows_last_end[ri] + min_gap):
                row_of_event.append(ri)
                rows_last_end[ri] = e
                placed = True
                break
        if not placed:
            row_of_event.append(len(rows_last_end))
            rows_last_end.append(e)
    events["__row__"] = row_of_event

    # --- Prepare SNPs (get 1-based position) ---
    snps = snps_df.copy()
    if "Pos_0based" in snps.columns:
        snps["__pos1__"] = snps["Pos_0based"].astype(int) + 1
    elif "Pos_1based" in snps.columns:
        snps["__pos1__"] = snps["Pos_1based"].astype(int)
    else:
        raise ValueError("SNPs DF must include 'Pos_0based' or 'Pos_1based'.")
    snps_in_region = snps[(snps["__pos1__"] >= region_start1) & (snps["__pos1__"] <= region_end1)].copy()

    # --- Layout and draw ---
    nrows = int(events["__row__"].max()) + 1
    for _, ev in events.iterrows():
        r = int(ev["__row__"])
        y = (nrows - 1 - r)  # row 0 at top
        x0 = float(ev["__clip_start__"])
        x1 = float(ev["__clip_end__"])

        # event rectangle
        width = max(0.0, x1 - x0)
        rect = Rectangle((x0, y - bar_height/2), width, bar_height,
                         facecolor=event_bar_color, edgecolor='none', alpha=0.9)
        ax.add_patch(rect)

        # SNP ticks (span full bar height)
        s_ev = snps_in_region[snps_in_region["EventID"] == ev["EventID"]]
        if not s_ev.empty:
            ymin = y - bar_height/2
            ymax = y + bar_height/2
            for _, rS in s_ev.iterrows():
                xv = float(rS["__pos1__"])
                nt = str(rS.get("Child_Call", "")).upper()[:1]
                color = nt_colors.get(nt, "black")
                ax.vlines(x=xv, ymin=ymin, ymax=ymax, lw=snp_line_width, color=color, alpha=0.95)

        # label above bar
        x_mid = (x0 + x1) / 2.0
        ax.text(x_mid, y + 0.7 * bar_height, str(ev["EventID"]),
                ha="center", va="bottom", fontsize=label_fontsize)

    # --- Cosmetics ---
    ax.set_xlim(region_start1, region_end1)
    ax.set_ylim(-0.8, nrows - 1 + 0.8)
    ax.set_yticks([])
    if show_xlabel:
        ax.set_xlabel("Genomic position (1-based)")

    # full integers on x-axis (no sci/offset)
    formatter = mticker.ScalarFormatter(useOffset=False)
    formatter.set_scientific(False)
    ax.xaxis.set_major_formatter(formatter)
    ax.ticklabel_format(style='plain', axis='x', useOffset=False)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=6, integer=True))

    ax.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.4)
    for spine in hide_spines:
        ax.spines[spine].set_visible(False)

    if add_legend:
        legend_elems = [Line2D([0], [0], color=nt_colors[k], lw=2, label=k)
                        for k in ["A","T","C","G"] if k in nt_colors]
        ax.legend(handles=legend_elems, title="Mut allele", fontsize=8, title_fontsize=9,
                  loc="upper right")

    if title is not None:
        ax.set_title(title, fontsize=10)

    summary = {
        "nrows": int(nrows),
        "events_plotted": events["EventID"].tolist(),
        "window": (region_start1, region_end1),
    }
    return ax, summary


In [141]:
from typing import Optional, Tuple, Dict
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

def plot_mut_epitope_region_with_gce_V2(
    Viz_Start: int,
    Viz_End: int,
    in_PerCodon_MutCt_DF,                 # cols: Pos_0based, Mutation_Count
    in_Genome_Graphic_Record,             # dna_features_viewer GraphicRecord (H37Rv)
    in_AllPeptides_GraphicRecord,         # GraphicRecord of all assayed peptides
    in_EpitopeCov_DF,                     # cols: start, end, Cov_PosEpitopes
    pGCE_df,          
    snps_df,
    *,
    mut_ylim: Optional[Tuple[float, float]] = (0, 15),
    figsize: Tuple[float, float] = (5, 5),
    height_ratios: Tuple[int, int, int, int, int] = (4, 5, 1, 4, 8),
    cmap_name: str = "Reds",
    point_size: float = 2.0,
    vline_lw: float = 0.9,
    epi_linewidth: float = 2.5,
    genome_label_threshold: int = 5,
    # GC-event track style
    gce_min_gap: int = 0,
    gce_bar_height: float = 0.5,
    gce_snp_line_width: float = 1.2,
    gce_label_fontsize: int = 8,
    show_bottom_xticks: bool = True,
    tight_layout: bool = True,
    epi_global_max: Optional[float] = None,   # if None, uses global max from in_EpitopeCov_DF
) -> Tuple[plt.Figure, Dict[str, plt.Axes]]:
    """
    Same as plot_mut_epitope_region, but adds a GC events track at the bottom.

    Axes order (top→bottom):
      0: Genome_Anno_ax (gene annotations)
      1: MF_ax          (mutation counts)
      2: EpiSumm_ax     (epitope density summary bar)
      3: AllAssayedPeptides_ax (all assayed peptides)
      4: GCE_ax         (gene conversion events)
    """
    # --- Crop graphic records to region ---
    Graphic_Record_cropped   = in_Genome_Graphic_Record.crop((Viz_Start, Viz_End + 1))
    AllPeptides_Records_cropped = in_AllPeptides_GraphicRecord.crop((Viz_Start, Viz_End + 1))

    # --- Create figure & axes ---
    fig, axs = plt.subplots(
        5, 1, figsize=figsize,
        gridspec_kw={'height_ratios': height_ratios}
    )
    Genome_Anno_ax, MF_ax, EpiSumm_ax, AllAssayedPeptides_ax, GCE_ax = axs

    # --- Gene Annotations (top track) ---
    if hasattr(Graphic_Record_cropped, "plot"):
        Graphic_Record_cropped.plot(strand_in_label_threshold = genome_label_threshold,
                                    ax=Genome_Anno_ax)
    Genome_Anno_ax.set_xlim(Viz_Start, Viz_End)
    Genome_Anno_ax.set_xticks([])

    # --- Mutation Counts ---
    if {"Pos_0based", "Mutation_Count"}.issubset(in_PerCodon_MutCt_DF.columns):
        mut_df = in_PerCodon_MutCt_DF.query(
            f"(Pos_0based + 1) >= {Viz_Start} & (Pos_0based + 1) <= {Viz_End}"
        ).copy()
        if not mut_df.empty:
            xvals = mut_df["Pos_0based"].to_numpy() + 1
            yvals = mut_df["Mutation_Count"].to_numpy()
            MF_ax.vlines(x=xvals, ymin=0, ymax=yvals, color="black", linewidth=vline_lw, alpha=0.6)
            MF_ax.scatter(x=xvals, y=yvals, color="red", s=point_size, alpha=1)

    MF_ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    MF_ax.ticklabel_format(style='plain', axis='x')
    MF_ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    MF_ax.set_xlim(Viz_Start, Viz_End)
    if mut_ylim is not None:
        MF_ax.set_ylim(*mut_ylim)
    MF_ax.set_xticks([])
    sns.despine(ax=MF_ax)

    # --- Epitope density summary (global-normalized) ---
    cmap = plt.get_cmap(cmap_name)
    if {"start", "end", "Cov_PosEpitopes"}.issubset(in_EpitopeCov_DF.columns):
        Reg_EpiCov_DF = in_EpitopeCov_DF.query(
            f"start >= {Viz_Start} & end <= {Viz_End}"
        ).copy()
        if not Reg_EpiCov_DF.empty:
            global_max = (in_EpitopeCov_DF["Cov_PosEpitopes"].max()
                          if epi_global_max is None else float(epi_global_max))
            norm = plt.Normalize(vmin=0, vmax=global_max if global_max > 0 else 1)
            for _, row in Reg_EpiCov_DF.iterrows():
                cov = row["Cov_PosEpitopes"]
                if cov > 0:
                    EpiSumm_ax.vlines(x=row["start"], ymin=0, ymax=1,
                                      color=cmap(norm(cov)), linewidth=epi_linewidth, alpha=1)

    EpiSumm_ax.set_xlim(Viz_Start, Viz_End)
    EpiSumm_ax.set_ylim(0, 1)
    EpiSumm_ax.set_xticks([])
    EpiSumm_ax.set_yticks([])
    for spine in ("left", "top", "right"):
        EpiSumm_ax.spines[spine].set_visible(False)
    EpiSumm_ax.set_xlabel("")


    # --- Assayed peptides ---
    if hasattr(AllPeptides_Records_cropped, "plot"):
        AllPeptides_Records_cropped.plot(strand_in_label_threshold=genome_label_threshold,
                                         ax=AllAssayedPeptides_ax)
    AllAssayedPeptides_ax.set_xlim(Viz_Start, Viz_End)
    AllAssayedPeptides_ax.set_ylim(-0.5, 5.5)
    AllAssayedPeptides_ax.set_xticks([])

    
    # --- GC events (bottom track) ---
    
    
    plot_multi_event_snp_packed_on_ax(GCE_ax,
                                      region_start1 = Viz_Start, region_end1 = Viz_End,
                                      pGCE_df = pGCE_DF, snps_df = snps_df, label_fontsize = 4,
                                      add_legend=False, )

    plot_multi_event_snp_packed_on_ax(GCE_ax,
                                      region_start1=Viz_Start, region_end1=Viz_End,
                                      pGCE_df = pGCE_df,
                                      snps_df = snps_df,
                                      min_gap=gce_min_gap,
                                      bar_height=gce_bar_height,
                                      snp_line_width=gce_snp_line_width,
                                      label_fontsize=gce_label_fontsize,
                                      #show_xlabel=show_xlabel_on_gce,
                                      add_legend=False,
                                      title=None,)



    if show_bottom_xticks:
        GCE_ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
        GCE_ax.ticklabel_format(style='plain', axis='x')
        GCE_ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))




    
    if tight_layout:
        fig.tight_layout()

    axes = {
        "Genome_Anno_ax": Genome_Anno_ax,
        "MF_ax": MF_ax,
        "EpiSumm_ax": EpiSumm_ax,
        "AllAssayedPeptides_ax": AllAssayedPeptides_ax,
        "GCE_ax": GCE_ax,
    }
    return fig, axes


### Define function for ploting Anno + GCEs in a region

In [142]:
from typing import Optional, Tuple, Dict
import matplotlib.pyplot as plt

def plot_Anno_with_GCE(
    Viz_Start: int,
    Viz_End: int,
    in_Genome_Graphic_Record,   # dna_features_viewer GraphicRecord (H37Rv)
    pGCE_df,                    # GC events dataframe
    snps_df,                    # SNPs dataframe (Pos_0based or Pos_1based, EventID, Child_Call)
    *,
    figsize: Tuple[float, float] = (5, 5), 
    dpi: int = 180,                            
    height_ratios: Tuple[float, float] = (1.0, 2.5),
    genome_label_threshold: int = 5,
    # GC-event track style
    gce_min_gap: int = 0,
    gce_bar_height: float = 0.5,
    gce_snp_line_width: float = 1.2,
    gce_label_fontsize: int = 8,
    show_xlabel_on_gce: bool = True,
    title: Optional[str] = None,
    tight_layout: bool = True,
) -> Tuple[plt.Figure, Dict[str, plt.Axes]]:
    """
    Plot gene annotations (top) and packed GC events with SNPs (bottom).
    Returns (fig, {'Genome_Anno_ax':..., 'GCE_ax':...}).
    """
    # Crop genome record to region
    Graphic_Record_cropped = in_Genome_Graphic_Record.crop((Viz_Start, Viz_End + 1))

    # Figure & axes
    fig, axs = plt.subplots(
        2, 1, figsize=figsize, dpi=dpi,
        gridspec_kw={'height_ratios': height_ratios}
    )
    Genome_Anno_ax, GCE_ax = axs

    # Top: gene annotations
    if hasattr(Graphic_Record_cropped, "plot"):
        Graphic_Record_cropped.plot(
            strand_in_label_threshold=genome_label_threshold, ax=Genome_Anno_ax
        )
    Genome_Anno_ax.set_xlim(Viz_Start, Viz_End)
    Genome_Anno_ax.set_xticks([])
    Genome_Anno_ax.set_xlabel("")

    # Bottom: packed GC events + SNPs (uses your on-axis helper)
    plot_multi_event_snp_packed_on_ax(
        GCE_ax,
        region_start1=Viz_Start, region_end1=Viz_End,
        pGCE_df=pGCE_df, snps_df=snps_df,
        min_gap=gce_min_gap,
        bar_height=gce_bar_height,
        snp_line_width=gce_snp_line_width,
        label_fontsize=gce_label_fontsize,
        show_xlabel=show_xlabel_on_gce,
        add_legend=False,
        title=None
    )

    if title:
        fig.suptitle(title, fontsize=11)

    if tight_layout:
        fig.tight_layout()

    return fig, {"Genome_Anno_ax": Genome_Anno_ax, "GCE_ax": GCE_ax}


# Parse `Mtb151CI` Isolate Metadata

In [143]:
Repo_DataDir = "../../Data"
InputAsmPath_Dir = f"{Repo_DataDir}/231121.InputAsmTSVs.MtbSetV3.151CI"

MtbSetV3_151CI_InputAsmPATHs_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAndSRAsm.FAPATHs.V1.tsv"
MtbSetV3_151CI_AsmSumm_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAsm.AsmSummary.V2.tsv"


### Reading in "WGA151CI_AsmSummary_DF"

In [144]:
WGA151CI_AsmSummary_DF = pd.read_csv(MtbSetV3_151CI_AsmSumm_TSV, sep = "\t")

SampleIDs_151CI_SOI = list( WGA151CI_AsmSummary_DF["SampleID"].values )
WGA151CI_SampleIDs = SampleIDs_151CI_SOI
WGA151CI_AsmSummary_DF.shape


(151, 7)

#### Create SampleID Mapping Dicts

In [145]:
WGA151CI_ID_To_PrimLineage_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'PrimaryLineage']].values)
WGA151CI_ID_To_SubLineage_Dict = dict( WGA151CI_AsmSummary_DF[["SampleID", "Lineage"]].values)
WGA151CI_ID_To_Dataset_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'Dataset_Tag']].values)  
ID_To_PrimLineage_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'PrimaryLineage']].values)
ID_To_SubLineage_Dict = dict( WGA151CI_AsmSummary_DF[["SampleID", "Lineage"]].values)
ID_To_Dataset_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'Dataset_Tag']].values)  


# Import/parse processed H37rv genome annotations

In [146]:
RepoRef_Dir = "../../References"

AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"
H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"

## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t")
H37Rv_GeneInfo_Subset_DF = H37Rv_GenomeAnno_Genes_DF[["H37rv_GeneID", "Symbol", "Feature", "Functional_Category", "Is_Pseudogene", "Product", "PEandPPE_Subfamily", "ExcludedGroup_Category"]]

RvID_To_Symbol_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['H37rv_GeneID', 'Symbol']].values)
Symbol_To_RvID_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'H37rv_GeneID']].values)
Symbol_To_FuncCat_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'Functional_Category']].values)

ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
Esx_Genes_DF = pd.read_csv(ESX_Genes_List_TSV, sep = '\t')

In [147]:
H37Rv_GenomeAnno_Genes_DF.head(1)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded


### Parse the WHO AR Variant Catalog (Mtb)

In [148]:
WHO_ResVar_DF = pd.read_csv("../../References/WHO_MtbAMR_Catalog/WHO_resistance_variants_all.csv", sep =",")
WHO_ResVar_LVL1_DF = WHO_ResVar_DF[ WHO_ResVar_DF["confidence"] == '1) Assoc w R']
WHO_ResVar_DF.shape

(17419, 5)

In [149]:
WHO_ResVar_LVL1_DF.shape

(201, 5)

In [150]:
WHO_ConfResGenes = list(WHO_ResVar_LVL1_DF["gene"].unique())
print(len(WHO_ConfResGenes))
print( WHO_ConfResGenes )

15
['inhA', 'rrs', 'eis', 'tlyA', 'embB', 'embA', 'ethA', 'katG', 'gyrA', 'rplC', 'gyrB', 'pncA', 'rpoB', 'rpsL', 'gid']


# Parse H37Rv Reference sequences (Genome, Genes, Proteins)

## Parse H37Rv genome sequence (DNA)

In [151]:
from Bio import SeqIO


In [152]:
H37rv_Ref_GBK_PATH = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.gbk"
H37Rv_FA = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.fasta"

H37Rv_Seq = SeqIO.read(H37Rv_FA, "fasta").seq
len(H37Rv_Seq)

4411532

## Parse H37Rv Protein (AA) and gene (DNA) sequences

In [153]:
O2_RefDir = "/n/data1/hms/dbmi/farhat/mm774/References"

MycoBrowser_RefFiles_Dir = f"{O2_RefDir}/190619_Mycobrowser_H37rv_ReferenceFiles"

H37Rv_Proteins_MycoBro_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"
H37Rv_Proteins_MycoBro_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.esxM_Added.fasta"
H37Rv_Proteins_NCBI_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"
H37RV_Genes_FA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_genes_v3.fasta"



H37Rv_FAA_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_proteins.faa"
H37Rv_FAA_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_proteins.esxM_Added.faa"
H37Rv_GBK_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_genomic.gbk"


In [154]:
!ls -1 $MycoBrowser_RefFiles_Dir

Mycobacterium_tuberculosis_H37Rv_genes_v3.fasta
Mycobacterium_tuberculosis_H37Rv_genome_v3.fasta
Mycobacterium_tuberculosis_H37Rv_genome_v3.fasta.fai
Mycobacterium_tuberculosis_H37Rv_gff_v3.gff
Mycobacterium_tuberculosis_H37Rv_gff_v3.REP13E12_Regions.gff
Mycobacterium_tuberculosis_H37Rv_proteins_v3.fasta
Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.esxM_Added.fasta
Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta
Mycobacterium_tuberculosis_H37Rv_txt_v3_PEPPE_subfamilies.txt
Mycobacterium_tuberculosis_H37Rv_txt_v3.txt.tsv


### Parse MycoBrowser Protein Seq Ref

In [155]:
dictOf_H37Rv_MycoBrow_ProtSeq = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_Proteins_MycoBro_FAA, "fasta"))):
    ShortID = record.name
    
    dictOf_H37Rv_MycoBrow_ProtSeq[ShortID] = record.seq


4091it [00:00, 143300.10it/s]


### Parse MycoBrowser Gene Seq Ref

In [156]:
dictOf_H37Rv_MycoBrow_Gene_Seq = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37RV_Genes_FA, "fasta"))):

    ShortID = record.name.split("|")[0]
    dictOf_H37Rv_MycoBrow_Gene_Seq[ShortID] = record.seq


4187it [00:00, 115084.51it/s]


In [157]:
list(dictOf_H37Rv_MycoBrow_Gene_Seq.keys())[:2]

['Rv3728', 'Rv3729']

### Parse NCBI Protein Seq Ref

In [158]:
dictOf_H37Rv_ProtSeq = {}
dictOf_H37Rv_ProtRecord = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_FAA_PATH, "fasta"))):
    Rec_Description = record.description
    dict_Attr = {}
    for i in Rec_Description.split(" "):
        ###Just looking for line with " " character (as key = value)
        if "=" in i:
            key = i.strip().split("=")[0].strip('"').strip('[')
            value = i.strip().split("=")[1].strip('"').strip(']')
            ###Put them in a dictionnary
            dict_Attr[key]=value
    
    ShortID = dict_Attr["locus_tag"]
    dictOf_H37Rv_ProtSeq[ShortID] = record.seq
    dictOf_H37Rv_ProtRecord[ShortID] = record


3907it [00:00, 99706.40it/s]


In [159]:
#dictOf_H37Rv_ProtSeq["Rv1196"]

In [160]:
#dictOf_H37Rv_ProtSeq["Rv1196"] == dictOf_H37Rv_MycoBrow_ProtSeq["Rv1196"]

In [161]:
dictOf_H37Rv_ProtSeq["Rv1792"]

Seq('MASRFMTDPHAMRDMAGRFEVHAQTVEDEARRMWASAQNISGAGWSGMAEATSLDTMT')

In [162]:
dictOf_H37Rv_MycoBrow_ProtSeq["Rv1792"]

Seq('MASRFMTDPHAMRDMAGRFEVHAQTVEDEARRMWASAQNISGAGWSGMAEATSLDTMT')

# Parse in H37Rv Homology-Map Results (k19w19)

### Define all HmMap file paths

In [163]:
Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9"

H37_Rv_MM2_HomologyMapping_Dir = f"{Main_Project_Dir}/250901.H37Rv.HomologyMapping.k19w19.ProcessedData.V2"

# Define paths to output TSVS

### Homologous regions (MERGED)
RvHmMap_Merged_ParaRegions_TSV  = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.ParalogousRegions.k19w19.tsv"
RvHmMap_Merged_LocalRepeats_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.LocalRepeats.k19w19.tsv"

### Homology map (pairwise alignments)
RvHmMap_Aln_All_TSV           = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.All.tsv"
RvHmMap_Aln_PRs_NoOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.tsv"
RvHmMap_Aln_LRs_WiOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.WiOverlap.tsv"

RvHmMap_Aln_PRs_NoOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.Clustered.tsv"
RvHmMap_Aln_LRs_WiOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.OnlyOverlap.Clustered.tsv"

### Variants from the homology map alignments
RvHmMap_Var_TSV      = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.tsv"
RvHmMap_Var_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.snps.tsv"


### Parse in HmRegions (`Paralogous_Regions` and `Local_Repeats`)

In [164]:
HmMapRegs_ParaRegs_k19w19_DF = pd.read_csv(RvHmMap_Merged_ParaRegions_TSV,
                                    sep="\t")
#HmMapRegs_ParaRegs_k19w19_DF["Overlap_Genes"] = HmMapRegs_ParaRegs_k19w19_DF["Overlap_Genes"].fillna("_")

HmMapRegs_ParaRegs_k19w19_DF.shape

(200, 13)

In [165]:
Rv_HHR_DF = HmMapRegs_ParaRegs_k19w19_DF
Rv_HHR_DF.shape

(200, 13)

In [166]:
HmMapRegs_LocalRepeats_k19w19_DF = pd.read_csv(RvHmMap_Merged_LocalRepeats_TSV, 
                                        sep="\t")

#HmMapRegs_LocalRepeats_k19w19_DF["Overlap_Genes"] = HmMapRegs_LocalRepeats_k19w19_DF["Overlap_Genes"].fillna("_")

HmMapRegs_LocalRepeats_k19w19_DF.shape

(50, 13)

In [167]:
HmMapRegs_All_LRsPRs_K19w19_DF = pd.concat([HmMapRegs_ParaRegs_k19w19_DF,
                                            HmMapRegs_LocalRepeats_k19w19_DF])

HmMapRegs_All_LRsPRs_K19w19_DF.shape

(250, 13)

In [168]:
HmMapRegs_ParaRegs_k19w19_DF.head(2)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,1,1,1,0,PR_HmRegion_000
1,1,NC_000962.3,80623,82664,81643.5,2041,"Rv0072,Rv0073",0,1,1,1,1,PR_HmRegion_001


#### Peak at head of each HmMap Regions DFs

In [169]:
HmMapRegs_ParaRegs_k19w19_DF.head()

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,1,1,1,0,PR_HmRegion_000
1,1,NC_000962.3,80623,82664,81643.5,2041,"Rv0072,Rv0073",0,1,1,1,1,PR_HmRegion_001
2,2,NC_000962.3,103705,105130,104417.5,1425,"Rv0094c,Rv0095c",0,2,2,2,2,PR_HmRegion_002
3,3,NC_000962.3,149571,149808,149689.5,237,PE_PGRS2,0,1,1,1,3,PR_HmRegion_003
4,4,NC_000962.3,177203,177447,177325.0,244,_,0,1,1,1,4,PR_HmRegion_004


In [170]:
HmMapRegs_LocalRepeats_k19w19_DF.head()

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID
0,0,NC_000962.3,333811,335879,334845.0,2068,PE_PGRS3,0,1,1,1,0,LR_HmRegion_000
1,1,NC_000962.3,366430,375121,370775.5,8691,"PPE5,PPE6",0,6,6,6,1,LR_HmRegion_001
2,2,NC_000962.3,424011,432951,428481.0,8940,"hspR,PPE7,PPE8",0,5,4,4,2,LR_HmRegion_002
3,3,NC_000962.3,566288,580814,573551.0,14526,"hbhA,Rv0476,Rv0477,deoC,Rv0479c,Rv0480c,Rv0481...",0,4,2,1,3,LR_HmRegion_003
4,4,NC_000962.3,631298,631436,631367.0,138,Rv0538,0,2,2,2,4,LR_HmRegion_004


### Parse in homology-map DFs (pairwise alignments between all homologous regions)

In [171]:
HmMap_Aln_k19w19_DF = pd.read_csv(RvHmMap_Aln_All_TSV,
                           sep="\t")
HmMap_Aln_k19w19_DF.shape

(776, 24)

In [172]:
HmMap_Aln_k19w19_NoOverlap_DF = pd.read_csv(RvHmMap_Aln_PRs_NoOverlap_Clustered_TSV,
                                     sep="\t")
HmMap_Aln_k19w19_NoOverlap_DF.shape

(640, 34)

In [173]:
HmMap_Aln_k19w19_LocalRepeat_DF = pd.read_csv(RvHmMap_Aln_LRs_WiOverlap_Clustered_TSV,
                                              sep="\t")
HmMap_Aln_k19w19_LocalRepeat_DF.shape

(136, 34)

In [174]:
HmMap_Aln_PR_ExactCopy_DF = HmMap_Aln_k19w19_NoOverlap_DF.query("SeqID == 1.0")
print(HmMap_Aln_PR_ExactCopy_DF.shape)

(255, 34)


In [175]:
HmMap_Aln_PR_NoPerfAln_DF = HmMap_Aln_k19w19_NoOverlap_DF.query("SeqID != 1.0")
print(HmMap_Aln_PR_NoPerfAln_DF.shape)

(385, 34)


In [176]:

HmMap_Aln_PR_MaxSeqID99_DF = HmMap_Aln_k19w19_NoOverlap_DF.query("SeqID <= 0.99")
print(HmMap_Aln_PR_MaxSeqID99_DF.shape)

(331, 34)


### Parse in HomologyMap Alignment Variants DFs

In [177]:
Mtb_HM_Var_DF = pd.read_csv(RvHmMap_Var_TSV, sep="\t")
Mtb_HM_Var_SNPs_DF = pd.read_csv(RvHmMap_Var_SNPs_TSV, sep="\t")

In [178]:
# Build trimmed + unique HM SNPs DF
UnqSNPs_TarCol = ['Query_Name', 'Query_Start', 'Query_End', 'Ref', 'Alt', 'SNP']

HM_Var_SNPs_TrimUnq_DF = Mtb_HM_Var_SNPs_DF[UnqSNPs_TarCol].drop_duplicates()
HM_Var_SNPs_TrimUnq_DF.shape

(51590, 6)

In [179]:
Mtb_HM_Var_DF.shape

(79508, 13)

In [180]:
Mtb_HM_Var_SNPs_DF.shape

(65617, 13)

# Parse curated TB antigens + epitopes (T-cell, CD4+)
Source Datasets: (Panda-24, Lindestam-16)

In [181]:
Repo_Epitope_MainDir = "../../Data/220813_MtbEpitopes"

#LPM_AllAssayedPeptides_Mapped_TSV = f"{Repo_Epitope_MainDir}/240820.Panda24_Lind16.Merged.PeptidesMappedToRv.AllAssayed.V1.tsv" 
Lind16_Peptides_HLAInfo_TSV = f"{Repo_Epitope_MainDir}/240815.Lindestram2016.HLA_ResponseInfo.V1.tsv" 

LPM_AllAssayedPeptides_WiMutInfo_TSV = f"{Repo_Epitope_MainDir}/240820.Panda24_Lind16.Merged.PeptidesMappedToRv.All.AnnoByMutFreq.V2.tsv" 

Rv_Genes_Epitope_SummStats_TSV = f"{Repo_Epitope_MainDir}/240820.RvGene.EpitopeMappingStats.V1.tsv"


In [182]:
!du -sh $Rv_Genes_Epitope_SummStats_TSV $LPM_AllAssayedPeptides_WiMutInfo_TSV

615K	../../Data/220813_MtbEpitopes/240820.RvGene.EpitopeMappingStats.V1.tsv
3.7M	../../Data/220813_MtbEpitopes/240820.Panda24_Lind16.Merged.PeptidesMappedToRv.All.AnnoByMutFreq.V2.tsv


### Parse in DF of all epitope mapping results (all assayed peptides, N = `18741`)

In [183]:
LPM_AllPep_V2_DF = pd.read_csv(LPM_AllAssayedPeptides_WiMutInfo_TSV, sep = "\t")
LPM_AllPep_V2_DF["Reactivity"] = LPM_AllPep_V2_DF["PosEpitope_Any"].replace(True, "Positive").replace(False, "Negative")
LPM_AllPep_V2_DF.shape

(18741, 31)

In [184]:
LPM_AllPep_V2_DF.columns

Index(['Epitope_ID', 'Epitope_Seq', 'Epitope_Len', 'RvID', 'Symbol', 'AA_Start', 'AA_End', 'Chrom', 'Rv_Start', 'Rv_End', 'EpitopeSeqFreqInAntigen', 'Dataset', 'Assayed_Panda24', 'PosEpitope_Panda24', 'Assayed_Lindestam16', 'PosEpitope_Lindestam16', 'PosEpitope_Any', 'EpitopeSymbol_ID', 'N_HmRegion', 'HasHmRegion', 'Antigen_LVL2', 'N_NSMut_Total', 'N_NSMut_mGCE', 'IsMutBymGCE', 'N_NSMut_pGCE', 'IsMutBypGCE', 'N_mGCEs_WiNS', 'WiNS_mGC_EventIDs', 'N_pGCEs_WiNS', 'WiNS_pGC_EventIDs', 'Reactivity'], dtype='object')

### Create a Dict that maps each epitope sequence to all gene's that contain the sequence

In [185]:
# Group by 'Epitope_Seq' and aggregate the unique 'Symbol' values into a string
EpiSeq_to_Symbols_dict = (
    LPM_AllPep_V2_DF.groupby('Epitope_Seq')['Symbol']
    .apply(lambda x: ', '.join(x.unique()))
    .to_dict()
)    

In [186]:
EpiSeq_to_Symbols_dict["DLYSKIESLPASQRD"]

'icd2'

### Subset DF for only POSITIVE EPITOPES or NEGATIVE PEPTIDES 

In [187]:
LPM_AllPep_V2_DF.shape

(18741, 31)

In [188]:
LPM_PosEpi_V2_DF = LPM_AllPep_V2_DF.query("PosEpitope_Any == True")
LPM_PosEpi_V2_DF.shape

(424, 31)

In [189]:
LPM_PosEpi_InL2Antigen_V2_DF = LPM_PosEpi_V2_DF.query("Antigen_LVL2 == True")
LPM_PosEpi_InL2Antigen_V2_DF.shape

(342, 31)

In [190]:
LPM_NegPep_V2_DF = LPM_AllPep_V2_DF.query("PosEpitope_Any == False")
LPM_NegPep_V2_DF.shape

(18317, 31)

## Parse gene-level epitope mapping summary for all H37Rv genes

In [191]:
Rv_Genes_EpitopeSummary_DF = pd.read_csv(Rv_Genes_Epitope_SummStats_TSV, sep="\t")
Rv_Genes_EpitopeSummary_DF.shape

(3841, 20)

In [192]:
Rv_Genes_EpitopeSummary_DF.head()

,Chrom,Start,End,Strand,Feature,H37rv_GeneID,Symbol,Functional_Category,Gene_Cat_V2,N_Pos,N_Neg,Total,Positive_Proportion,Middle,Length,AnyPosEpitope,Antigen_LVL2,N_HmRegion,HasHmRegion,AntigenLVL2_And_HHR_Comb
0,NC_000962.3,0,1524,+,CDS,Rv0001,dnaA,information pathways,information pathways,0,8,8,0.0,762.0,1524,False,False,0,False,NonReactive-Unq
1,NC_000962.3,2051,3260,+,CDS,Rv0002,dnaN,information pathways,information pathways,0,6,6,0.0,2655.5,1209,False,False,0,False,NonReactive-Unq
2,NC_000962.3,3279,4437,+,CDS,Rv0003,recF,information pathways,information pathways,0,6,6,0.0,3858.0,1158,False,False,0,False,NonReactive-Unq
3,NC_000962.3,4433,4997,+,CDS,Rv0004,Rv0004,conserved hypotheticals,conserved hypotheticals,0,3,3,0.0,4715.0,564,False,False,0,False,NonReactive-Unq
4,NC_000962.3,5239,7267,+,CDS,Rv0005,gyrB,information pathways,information pathways,0,8,8,0.0,6253.0,2028,False,False,0,False,NonReactive-Unq


In [193]:
Rv_AntigensL2_EpitopeSummary_DF = Rv_Genes_EpitopeSummary_DF.query("Antigen_LVL2 == True")

Antigens_LVL2 = Rv_AntigensL2_EpitopeSummary_DF["Symbol"].values

print(len(Antigens_LVL2))
Rv_AntigensL2_EpitopeSummary_DF.shape

53


(53, 20)

In [194]:
Antigens_LVL2[:5]

array(['Rv0010c', 'pepA', 'Rv0176', 'Rv0276', 'eccC3'], dtype=object)

### `Lind-16` - parse and process HLA and epitope positivity info in 63 ZA individuals

In [195]:
Lind16_IEDB_AllPeptides_HLAInfo_DF = pd.read_csv(Lind16_Peptides_HLAInfo_TSV, sep = "\t")

Lind16_IEDB_AllPeptides_HLAInfo_DF["Gene(s)"] = Lind16_IEDB_AllPeptides_HLAInfo_DF["Epitope_Seq"].map(EpiSeq_to_Symbols_dict)

Lind16_IEDB_PosEpitopes_HLAInfo_DF = Lind16_IEDB_AllPeptides_HLAInfo_DF.query("Epitope_Status == 'Positive' ")


# Group by "Epitope_Seq" and sum the relevant numeric columns
Lind16_EpiPosFreq_DF = Lind16_IEDB_AllPeptides_HLAInfo_DF.groupby("Epitope_Seq")["Num_Positive"].sum()
Lind16_EpiPosFreq_DF = Lind16_EpiPosFreq_DF.reset_index()

Lind16_EpiPosFreq_DF["Num_Assayed"] = 63

# Assuming Num_Assayed is 63 for each epitope, calculate the fraction of individuals positive for each epitope
Lind16_EpiPosFreq_DF['Fraction_Positive'] = Lind16_EpiPosFreq_DF['Num_Positive'] / 63

Lind16_EpiPosFreq_DF["Gene(s)"] = Lind16_EpiPosFreq_DF["Epitope_Seq"].map(EpiSeq_to_Symbols_dict)

Lind16_EpiPosFreq_DF["Gene(s)"] = Lind16_EpiPosFreq_DF["Gene(s)"].fillna("")


In [196]:
Lind16_IEDB_AllPeptides_HLAInfo_DF.head(3)

,Epitope_Seq,Epitope_Status,HLA_Allele,Num_Assayed,Num_Positive,Fraction_PositiveAssay,Gene(s)
0,DAHGAMIRAQAGSLE,Positive,HLA-DQB1*06:02,5,5,1.0,"esxI, esxV"
1,ISTNIRQAGVQYSRA,Positive,HLA-DQB1*06:02,4,4,1.0,esxB
2,MHVSFVMAYPEMLAA,Positive,HLA-DQB1*06:02,4,4,1.0,NaN


In [197]:
Lind16_IEDB_AllPeptides_HLAInfo_DF.shape

(1025, 7)

In [198]:
Lind16_IEDB_PosEpitopes_HLAInfo_DF.shape

(499, 7)

In [199]:
Lind16_EpiPosFreq_DF["Epitope_Seq"].nunique()

761

In [200]:
Lind16_IEDB_AllPeptides_HLAInfo_DF["Epitope_Seq"].nunique()

761

In [201]:
Lind16_IEDB_PosEpitopes_HLAInfo_DF["Epitope_Seq"].nunique()

235

# Parse gene-level epitope + mutation + GCE stats - (antigen/epitope info, GCE info, Mutational Burden, etc)

In [202]:
Repo_AntigenMut_MainDir = "../../Data/250609.AntigenMutationalBurdenAnalysis"

Rv_Gene_MutStats_V1_TSV     = f"{Repo_AntigenMut_MainDir}/250808.RvGenes.GCE.MutationFreq.Epitopes.SummaryStats.V1.tsv"

Rv_Antigens_MutStats_V1_TSV = f"{Repo_AntigenMut_MainDir}/250808.RvL2Antigens.GCE.MutationFreq.Epitopes.SummaryStats.V1.tsv"

PerRvPosition_EpitopeCovStats_TSV = f"{Repo_AntigenMut_MainDir}/250609.PerRvPos.EpitopeCoverage.AssayedPosOnly.V1.tsv.gz"

PerRvCodon_MutationFreq_TSV = f"{Repo_AntigenMut_MainDir}/250609.PerRvCodon.MissenseMutatationFreq.V1.tsv.gz"

### a) Parse gene-level - mutational burden and epitope mapping summary DFs

In [203]:
Genes_GCandMutFreqStats_DF = pd.read_csv(Rv_Gene_MutStats_V1_TSV,
                                           sep = "\t")

Genes_GCandMutFreqStats_DF.shape

(3841, 36)

In [204]:
Antigen_GCandMutFreqStats_DF = pd.read_csv(Rv_Antigens_MutStats_V1_TSV,
                                           sep = "\t")

Antigen_GCandMutFreqStats_DF.shape

(53, 38)

In [205]:
Antigen_GCandMutFreqStats_DF.head()

,Chrom,Start,End,Strand,Feature,H37rv_GeneID,Symbol,Functional_Category,Gene_Cat_V2,N_Pos,N_Neg,Total,Positive_Proportion,Middle,Length,AnyPosEpitope,Antigen_LVL2,N_HmRegion,HasHmRegion,AntigenLVL2_And_HHR_Comb,pGCE_Ovrlap,mGCE_Ovrlap,Total_NS_Muts,Total_NS_Muts_InEpitope,Total_NS_Muts_BymGCE,Total_NS_Muts_BypGCE,Total_NS_Muts_BymGCE_InEpitope,Total_NS_Muts_BypGCE_InEpitope,RelFreq_NS_Muts,RelFreq_NS_Mut_InEpitope,Has_mGCE,Has_mGCE_WiNSMut,Has_mGCE_WiNSMut_InEpitope,Has_pGCE,Has_pGCE_WiNSMut,Has_pGCE_WiNSMut_InEpitope,Length_WiPosEpitope,RelLength_WiPosEpitope
0,NC_000962.3,3894425,3895607,+,CDS,Rv3478,PPE60,PE/PPE,PE/PPE,5,15,20,0.250000,3895016.0,1182,True,True,1,True,Antigen-HHR,8,8,141.0,24.0,120.0,120.0,120.0,141.0,35.786802,6.091371,True,True,True,True,True,True,165,0.139594
1,NC_000962.3,1341005,1341290,+,CDS,Rv1198,esxL,cell wall and cell processes,esx,9,7,16,0.562500,1341147.5,285,True,True,1,True,Antigen-HHR,10,7,30.0,27.0,13.0,20.0,13.0,30.0,31.578947,28.421053,True,True,True,True,True,True,225,0.789474
2,NC_000962.3,2626222,2626519,-,CDS,Rv2347c,esxP,cell wall and cell processes,esx,10,6,16,0.625000,2626370.5,297,True,True,1,True,Antigen-HHR,9,9,17.0,2.0,12.0,12.0,12.0,17.0,17.171717,2.020202,True,True,True,True,True,True,210,0.707071
3,NC_000962.3,1339348,1340524,+,CDS,Rv1196,PPE18,PE/PPE,PE/PPE,27,56,83,0.325301,1339936.0,1176,True,True,2,True,Antigen-HHR,8,7,61.0,44.0,45.0,46.0,45.0,61.0,15.561224,11.224490,True,True,True,True,True,True,705,0.599490
4,NC_000962.3,1532442,1533633,-,CDS,Rv1361c,PPE19,PE/PPE,PE/PPE,16,21,37,0.432432,1533037.5,1191,True,True,1,True,Antigen-HHR,5,5,61.0,9.0,36.0,36.0,36.0,61.0,15.365239,2.267003,True,True,True,True,True,True,390,0.327456


In [206]:
Antigen_GCandMutFreqStats_DF["Length"].sum()

48474

#### How much oftthe genome is an epitope genome-wide (HHR + Unique)

In [207]:
Antigen_GCandMutFreqStats_DF["Length_WiPosEpitope"].sum()

9117

#### How much of genome is an epitope in an HHR?

In [208]:
Antigen_GCandMutFreqStats_DF.query("AntigenLVL2_And_HHR_Comb == 'Antigen-HHR'")["Length_WiPosEpitope"].sum()


4395

#### How much of genome is an HHR?

In [209]:
Rv_HHR_DF["Length"].sum()

256568

In [210]:
4411532

4411532

#### How much of genome is an HHR-antigen?

In [211]:
Antigen_GCandMutFreqStats_DF.query("AntigenLVL2_And_HHR_Comb == 'Antigen-HHR'")["Length"].sum()

14304

In [212]:
Genes_GCandMutFreqStats_DF.query("AntigenLVL2_And_HHR_Comb == 'Antigen-Unq'")["Length"].sum()

34170

In [213]:
Genes_GCandMutFreqStats_DF.query("AntigenLVL2_And_HHR_Comb == 'NonReactive-HHR'")["Length"].sum()

326052

In [214]:
Genes_GCandMutFreqStats_DF["AntigenLVL2_And_HHR_Comb"].value_counts()

AntigenLVL2_And_HHR_Comb
NonReactive-Unq    3584
NonReactive-HHR     204
Antigen-Unq          33
Antigen-HHR          20
Name: count, dtype: int64

In [215]:
# Antigen_GCandMutFreqStats_DF = Genes_GCandMutFreqStats_DF.query("Antigen_LVL2 == True")
# Antigen_GCandMutFreqStats_DF.shape

### b) Parse Per Codon Position Mutation Freq (Across Gubbins ASR SNP Events) + Epitope mapping coverage DFs

In [216]:
RvPerCodon_NSMutFreq_DF = pd.read_csv(PerRvCodon_MutationFreq_TSV,
                                  sep = "\t")
RvPerCodon_NSMutFreq_DF.shape 

(12132, 4)

In [217]:
RvPerCodon_NSMutFreq_DF.head(1)

,Symbol,Codon,Mutation_Count,Pos_0based
0,35kd_ag,230.0,1,3057374


### c) Parse Per Genome Position  Epitope mapping coverage DF

In [218]:
Rv_EpitopeCov_NonZero_DF = pd.read_csv(PerRvPosition_EpitopeCovStats_TSV,
                                  sep = "\t")

Rv_EpitopeCov_NonZero_DF.shape

(746901, 5)

In [219]:
Rv_EpitopeCov_NonZero_DF.head(2)

,chrom,start,end,Cov_PosEpitopes,Cov_AssayedPeptides
0,NC_000962.3,537,538,0,1
1,NC_000962.3,538,539,0,1


# Parse `Mtb151CI` Gubbins Results

### Define dictionary of file paths for Gubbins analysis

In [220]:
AnalysisName = "250901.WGA151CI.V9"

Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9"

Target_Output_Dir = f"{Main_Project_Dir}/{AnalysisName}"

WGA151_Gubbins_V1_OutputDir = f"{Target_Output_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"

Gubbins_V1_OutputDir = WGA151_Gubbins_V1_OutputDir

WGA151_Gubbins_OutPrefix = "Gubbins"

WGA151_Gubbins_FullPrefix_PATH  = f"{WGA151_Gubbins_V1_OutputDir}/{WGA151_Gubbins_OutPrefix}"

WGA151_Gubbins_FilePath_Dict = {}

WGA151_Gubbins_FilePath_Dict["NodeLabelledTree_PATH"]           = f"{WGA151_Gubbins_FullPrefix_PATH}.node_labelled.final_tree.tre"
WGA151_Gubbins_FilePath_Dict["NodeToPriLineage_Dict_JSON"]      = f"{WGA151_Gubbins_FullPrefix_PATH}.NodeToPrimaryLineage.json"
WGA151_Gubbins_FilePath_Dict["BranchStats_CSV"]                 = f"{WGA151_Gubbins_FullPrefix_PATH}.per_branch_statistics.csv"
WGA151_Gubbins_FilePath_Dict["BranchStats_WithLineage_CSV"]     = f"{WGA151_Gubbins_FullPrefix_PATH}.per_branch_statistics.WithLineagePerNode.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_1kb_H37Rv_TSV"]         = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPer1kb.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"]        = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerGene.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"]         = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerMergedHomologousRegion.tsv"

WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]             = f"{WGA151_Gubbins_FullPrefix_PATH}.recombination_predictions.Anno.tsv"
WGA151_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"]              = f"{WGA151_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.All.tsv"  
WGA151_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"]       = f"{WGA151_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv" 

WGA151_Gubbins_FilePath_Dict["ASR_SNPs_Anno_V2_All_TSV"]        = f"{WGA151_Gubbins_FullPrefix_PATH}.SNPs.AnnoByEvent.AnnoByCDSEffect.All.tsv"
WGA151_Gubbins_FilePath_Dict["ASR_SNPs_Anno_V2_EventsOnly_TSV"] = f"{WGA151_Gubbins_FullPrefix_PATH}.SNPs.AnnoByEvent.AnnoByCDSEffect.EventSNPsOnly.tsv"

WGA151_Gubbins_FilePath_Dict["BaseAncRec_All_WiCDS_Epitope_Info_TSV"] = f"{WGA151_Gubbins_FullPrefix_PATH}.SNPs.AnnoByEvent.AnnoByCDSandEpitope.All.V2.tsv"  


RecombPreds_Anno_ByHmMatch_ByEpiMut_V3_TSV = f"{Gubbins_V1_OutputDir}/Gubbins.All_GCEs.AnnoBy.EpitopeEffect.V3.tsv"

WGA151_Gubbins_FilePath_Dict["GCEvents.AnnoByHmMatch.AnnoByEpitopeEffect.V3"] = RecombPreds_Anno_ByHmMatch_ByEpiMut_V3_TSV



######### Add Event to Paralog Mapping Results File Paths ##########

EventMapping_ResultsDir = f"{Target_Output_Dir}/RecombEvent-To-HmRegion-Comparison-V3"

GRE_Anno_ByTopHomologMatch_TSV              = f"{EventMapping_ResultsDir}/GubbinsEvents.WiParalogMapping.V1.tsv"
EventMappingToAllParalogs_Info_TSV_PATH     = f"{EventMapping_ResultsDir}/EventMapping.EventsToAllHmRegions.SeqComparisonInfo.tsv"
Pickle_PATH_dictOf_EventAndHomolog_KmerComp = f"{EventMapping_ResultsDir}/EventMapping.DictOf.KmerComparisons.pickle"   


WGA151_Gubbins_FilePath_Dict["GCEvents_WiParalogMapInfo_TSV"]       = GRE_Anno_ByTopHomologMatch_TSV
WGA151_Gubbins_FilePath_Dict["EventMappingToAllParalogs_TSV"]       = EventMappingToAllParalogs_Info_TSV_PATH
WGA151_Gubbins_FilePath_Dict["EventKmerAnalysis_Dict_PicklePath"]   = Pickle_PATH_dictOf_EventAndHomolog_KmerComp

HmRegions_MappedEvents_TSV = f"{EventMapping_ResultsDir}/GCE.Stats.PerMergedHmRegion.PRs.tsv"
HmPairs_MappedEvents_TSV   = f"{EventMapping_ResultsDir}/GCE.Stats.PerPairwiseAln.PRs.tsv"

WGA151_Gubbins_FilePath_Dict["MergedHmRegion_PRs_GCE_Stats_TSV"] = HmRegions_MappedEvents_TSV
WGA151_Gubbins_FilePath_Dict["HmMapAln_PRs_GCE_Stats_TSV"]       = HmPairs_MappedEvents_TSV

####################################################################


## Parse Gene Conv Event Info (pGCEs + mGCEs)

In [221]:
pGCE_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["GCEvents.AnnoByHmMatch.AnnoByEpitopeEffect.V3"],
                         sep = "\t")

pGCE_DF.shape

(324, 50)

In [222]:
pGCE_EventIDs = pGCE_DF["EventID"].unique()
len(pGCE_EventIDs)

324

#### Filter All putative GC events to MAPPED GC events

In [223]:
mGCE_DF = pGCE_DF.query("(Max_KmerMatch_ToHm > 0.5)")
mGCE_DF.shape

(213, 50)

In [224]:
mGCE_EventIDs = mGCE_DF["EventID"].unique()
len(mGCE_EventIDs)

213

In [225]:
pGCE_DF.head(1)

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs,N_LCR_Ovrlap,OvrlapWi_LowComplexityRegion,N_LowPmap_Ovrlap,OvrlapWi_LowPmap,LenOfTaxaList,IsTermNode,Freq_SNP_FoundInAnyPR,NumHmTargets,NumHm_AtMaxKmerMatch,Top_KmerMatch_HomologTarIDs,Top_KmerMatch_HomologGeneIDs,Max_KmerMatch_ToHm,DistToHm_TopKmatch,TotalKmers_Eval,NumHm_AtMaxSNPMatch,Top_SNPMatch_HomologTarIDs,Top_SNPMatch_HomologGeneIDs,Max_SNPMatch_ToHm,DistToHm_TopSNPmatch,Overlap_WiAntigenLVL2Gene,N_NS_Mut,N_EpiMutated_By_GCE,N_NegPepMutated_By_GCE,EpitopeIDs_NSMutByEvent
0,NC_000962.3,103600,104478,0.0,.,Node_148,Node_133,1737.432787,10,"['mada_2-31', 'mada_1-41', 'MT_0080', 'mada_10...",103599,104038.5,879,NaN,"Rv0093c,Rv0094c","Rv0093c,Rv0094c",False,False,True,False,Event_001,2,1,0,0,PR_HmRegion_002,0,0,1,1,130,False,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.8608,NaN,79,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,False,0,0,0,.


## Parse Gubbins ASR SNP files (w/ Codon Variant Annotations)

In [226]:
Gub_SNPs_V2_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["BaseAncRec_All_WiCDS_Epitope_Info_TSV"],
                                         sep = "\t")

Gub_SNPs_V2_DF.shape

(26508, 23)

In [227]:
Gub_SNPs_V2_DF.head()

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,EventID,Pos_0based,Chrom,NumHmOvrlap,HmOvrlap,RegionType,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Start,End,Lineage,N_Epitopes_Overlap
0,1088,Node_1,N1176,G,A,NaN,1087,NC_000962.3,0,0,Unq,1,dnaA,+,1087.0,363.0,2.0,S,N,1087,1088,lineage5,0
1,10321,Node_1,N1176,C,T,NaN,10320,NC_000962.3,0,0,Unq,1,Rv0007,+,407.0,136.0,3.0,NaN,NaN,10320,10321,lineage5,0
2,11846,Node_1,N1176,C,G,NaN,11845,NC_000962.3,0,0,Unq,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11845,11846,lineage5,0
3,16515,Node_1,N1176,T,C,NaN,16514,NC_000962.3,0,0,Unq,1,pknB,-,955.0,319.0,2.0,D,G,16514,16515,lineage5,0
4,34063,Node_1,N1176,C,T,NaN,34062,NC_000962.3,0,0,Unq,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34062,34063,lineage5,0


### Subset Gubbins ASR SNPs (V2 Table) into groups of interest

In [228]:
Gub_SNPs_NSMutOnly_DF       = Gub_SNPs_V2_DF[~Gub_SNPs_V2_DF["Mut_AA"].isna()].query("Ref_AA != Mut_AA")

Gub_SNPs_NSMutIn_pGCE_DF    = Gub_SNPs_NSMutOnly_DF.query("EventID != 'None'")

Gub_SNPs_NSMutIn_mGCE_DF    = Gub_SNPs_NSMutOnly_DF[ Gub_SNPs_NSMutOnly_DF["EventID"].isin(mGCE_EventIDs) ]

Gub_SNPs_NSMutInEpitope_DF  = Gub_SNPs_NSMutOnly_DF.query(" N_Epitopes_Overlap > 0 ")


# Begin Analysis

# Define PPE18 protein sequence (From H37Rv reference)

#### Pull out PPE18 (Rv1196) protein sequence

In [229]:
PPE18_AA_Seq = dictOf_H37Rv_MycoBrow_ProtSeq["Rv1196"]
len(PPE18_AA_Seq)

391

In [230]:
PPE18_AA_Seq

Seq('MVDFGALPPEINSARMYAGPGSASLVAAAQMWDSVASDLFSAASAFQSVVWGLT...AAG')

In [231]:
# Output H37Rv PPE18 sequence to .faa file
write_AAseq_to_fasta(PPE18_AA_Seq, f"H37Rv_PPE18", f"./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.H37Rv.AASeq.faa")    


In [232]:
!head ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.H37Rv.AASeq.faa

>H37Rv_PPE18
MVDFGALPPEINSARMYAGPGSASLVAAAQMWDSVASDLFSAASAFQSVVWGLTVGSWIG
SSAGLMVAAASPYVAWMSVTAGQAELTAAQVRVAAAAYETAYGLTVPPPVIAENRAELMI
LIATNLLGQNTPAIAVNEAEYGEMWAQDAAAMFGYAAATATATATLLPFEEAPEMTSAGG
LLEQAAAVEEASDTAAANQLMNNVPQALQQLAQPTQGTTPSSKLGGLWKTVSPHRSPISN
MVSMANNHMSMTNSGVSMTNTLSSMLKGFAPAAAAQAVQTAAQNGVRAMSSLGSSLGSSG
LGGGVAANLGRAASVGSLSVPQAWAAANQAVTPAARALPLTSLTSAAERGPGQMLGGLPV
GQMGARAGGGLSGVLRVPPRPYVMPHSPAAG


## Create DF of all events in `PPE18`

In [233]:
GCE_InPPE18_DF = pGCE_DF.query("Overlap_Genes == 'PPE18'")
GCE_InPPE18_DF.shape

(7, 50)

In [234]:
EventIDs_InPPE18 = list(GCE_InPPE18_DF["EventID"].values)
EventIDs_InPPE18

['Event_091',
 'Event_092',
 'Event_093',
 'Event_094',
 'Event_095',
 'Event_096',
 'Event_097']

In [235]:
i_PPE18_SNPs_NSMut_DF = Gub_SNPs_NSMutOnly_DF.query(f"EventID == 'Event_094' ")

In [236]:
i_PPE18_UnqAASubs_DF = make_unq_aa_subs_df(i_PPE18_SNPs_NSMut_DF)

# Infer recombinant `PPE18` Protein Sequence for each GC Event

In [237]:
!mkdir ./PPE18.AllGCEvent.RecombinantProteinSeqs


mkdir: cannot create directory ‘./PPE18.AllGCEvent.RecombinantProteinSeqs’: File exists


In [238]:
for i_EventID in EventIDs_InPPE18:

    #print(i_EventID)
    i_PPE18_SNPs_NSMut_DF = Gub_SNPs_NSMutOnly_DF.query(f"EventID == '{i_EventID}' ")
    #print(i_PPE18_SNPs_NSMut_DF.shape)

    i_PPE18_UnqAASubs_DF = make_unq_aa_subs_df(i_PPE18_SNPs_NSMut_DF)
    print(f"{i_EventID} - # of missense mutations:",i_PPE18_UnqAASubs_DF.shape[0])

    Event_PPE18_AASeq, _ = apply_aa_substitutions(PPE18_AA_Seq, i_PPE18_UnqAASubs_DF)
    print("\nLength of resulting mutated PPE18 seq:", len(Event_PPE18_AASeq))
    
    write_AAseq_to_fasta(Event_PPE18_AASeq, f"PPE18_{i_EventID}_RecombinantAASeq", f"./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.{i_EventID}.GCE_AASeq.faa")

    print("-----\n\n")


Event_091 - # of missense mutations: 1
Mutated position 29: Q → K

Length of resulting mutated PPE18 seq: 391
-----


Event_092 - # of missense mutations: 2
Mutated position 29: Q → K
Mutated position 54: V → T

Length of resulting mutated PPE18 seq: 391
-----


Event_093 - # of missense mutations: 1
Mutated position 29: Q → K

Length of resulting mutated PPE18 seq: 391
-----


Event_094 - # of missense mutations: 13
Mutated position 102: G → R
Mutated position 115: A → T
Mutated position 119: I → T
Mutated position 121: I → T
Mutated position 134: A → E
Mutated position 135: V → A
Mutated position 137: E → Q
Mutated position 139: E → A
Mutated position 141: G → S
Mutated position 142: E → Q
Mutated position 145: A → G
Mutated position 149: A → E
Mutated position 152: F → Y

Length of resulting mutated PPE18 seq: 391
-----


Event_095 - # of missense mutations: 6
Mutated position 170: E → D
Mutated position 173: E → L
Mutated position 174: M → I
Mutated position 176: S → N
Mutated posi

### Peak at output directory

In [239]:
!ls -1 ./PPE18.AllGCEvent.RecombinantProteinSeqs

PPE18.Event_091.GCE_AASeq.faa
PPE18.Event_092.GCE_AASeq.faa
PPE18.Event_093.GCE_AASeq.faa
PPE18.Event_094.GCE_AASeq.faa
PPE18.Event_095.GCE_AASeq.faa
PPE18.Event_096.GCE_AASeq.faa
PPE18.Event_097.GCE_AASeq.faa
PPE18.H37Rv.AASeq.faa


In [242]:
!wc -l ./PPE18.AllGCEvent.RecombinantProteinSeqs/*

   8 ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.Event_091.GCE_AASeq.faa
   8 ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.Event_092.GCE_AASeq.faa
   8 ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.Event_093.GCE_AASeq.faa
   8 ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.Event_094.GCE_AASeq.faa
   8 ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.Event_095.GCE_AASeq.faa
   8 ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.Event_096.GCE_AASeq.faa
   8 ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.Event_097.GCE_AASeq.faa
   8 ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.H37Rv.AASeq.faa
  64 total


In [245]:
!head ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.H37Rv.AASeq.faa

>H37Rv_PPE18
MVDFGALPPEINSARMYAGPGSASLVAAAQMWDSVASDLFSAASAFQSVVWGLTVGSWIG
SSAGLMVAAASPYVAWMSVTAGQAELTAAQVRVAAAAYETAYGLTVPPPVIAENRAELMI
LIATNLLGQNTPAIAVNEAEYGEMWAQDAAAMFGYAAATATATATLLPFEEAPEMTSAGG
LLEQAAAVEEASDTAAANQLMNNVPQALQQLAQPTQGTTPSSKLGGLWKTVSPHRSPISN
MVSMANNHMSMTNSGVSMTNTLSSMLKGFAPAAAAQAVQTAAQNGVRAMSSLGSSLGSSG
LGGGVAANLGRAASVGSLSVPQAWAAANQAVTPAARALPLTSLTSAAERGPGQMLGGLPV
GQMGARAGGGLSGVLRVPPRPYVMPHSPAAG


In [244]:
!head ./PPE18.AllGCEvent.RecombinantProteinSeqs/PPE18.Event_097.GCE_AASeq.faa

>PPE18_Event_097_RecombinantAASeq
MVDFGALPPEINSARMYAGPGSASLVAAAQMWDSVASDLFSAASAFQSVVWGLTVGSWIG
SSAGLMVAAASPYVAWMSVTAGQAELTAAQVRVAAAAYETAYGLTVPPPVIAENRAELMI
LIATNLLGQNTPAIAVNEAEYGEMWAQDAAAMFGYAAATATATATLLPFEEAPEMTSAGG
LLEQAAAVEEASDTAAANQLMNNVPQALQQLAQPAQGVVPSSKLGGLWTAVSPHRSPISN
MVSMANNHMSMTNSGVSMTNTLSSMLKGFAPAAAAQAVQTAAQNGVRAMSSLGSSLGSSG
LGGGVAANLGRAASVGSLSVPQAWAAANQAVTPAARALPLTSLTSAAERGPGQMLGGLPV
GQMGARAGGGLSGVLRVPPRPYVMPHSPAAG


# Extras

In [113]:
Gub_SNPs_V2_DF.head()

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,EventID,Pos_0based,Chrom,NumHmOvrlap,HmOvrlap,RegionType,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Start,End,Lineage,N_Epitopes_Overlap
0,1088,Node_1,N1176,G,A,NaN,1087,NC_000962.3,0,0,Unq,1,dnaA,+,1087.0,363.0,2.0,S,N,1087,1088,lineage5,0
1,10321,Node_1,N1176,C,T,NaN,10320,NC_000962.3,0,0,Unq,1,Rv0007,+,407.0,136.0,3.0,NaN,NaN,10320,10321,lineage5,0
2,11846,Node_1,N1176,C,G,NaN,11845,NC_000962.3,0,0,Unq,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11845,11846,lineage5,0
3,16515,Node_1,N1176,T,C,NaN,16514,NC_000962.3,0,0,Unq,1,pknB,-,955.0,319.0,2.0,D,G,16514,16515,lineage5,0
4,34063,Node_1,N1176,C,T,NaN,34062,NC_000962.3,0,0,Unq,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34062,34063,lineage5,0


In [135]:
Event_091_PPE18_SNPs_NSMut_DF = Gub_SNPs_NSMutOnly_DF.query("EventID == 'Event_091' ")
Event_091_PPE18_SNPs_NSMut_DF.shape

(1, 23)

In [136]:
Event_092_PPE18_SNPs_NSMut_DF = Gub_SNPs_NSMutOnly_DF.query("EventID == 'Event_092' ")

Event_092_PPE18_SNPs_NSMut_DF.shape

(3, 23)

In [137]:
Event_093_PPE18_SNPs_NSMut_DF = Gub_SNPs_V2_DF.query("EventID == 'Event_093' ")
Event_093_PPE18_SNPs_NSMut_DF.shape

(5, 23)

In [138]:
Event_094_PPE18_SNPs_NSMut_DF = Gub_SNPs_V2_DF.query("EventID == 'Event_094' ")
Event_094_PPE18_SNPs_NSMut_DF.shape

(20, 23)

In [139]:
Event_095_PPE18_SNPs_NSMut_DF = Gub_SNPs_V2_DF.query("EventID == 'Event_095' ")
Event_095_PPE18_SNPs_NSMut_DF.shape

(12, 23)

In [140]:
Event_096_PPE18_SNPs_NSMut_DF = Gub_SNPs_V2_DF.query("EventID == 'Event_096' ")
Event_096_PPE18_SNPs_NSMut_DF.shape

(8, 23)

In [142]:
Event_097_PPE18_SNPs_NSMut_DF = Gub_SNPs_V2_DF.query("EventID == 'Event_097' ")
Event_097_PPE18_SNPs_NSMut_DF.shape

(11, 23)

## 1st Attempt: Create GCE-PPE18 sequence - `Event_091` in `PPE18'

In [219]:
Event_091_PPE18_SNPs_NSMut_DF

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,EventID,Pos_0based,Chrom,NumHmOvrlap,HmOvrlap,RegionType,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Start,End,Lineage,N_Epitopes_Overlap
15397,1339436,Node_36,Node_28,C,A,Event_091,1339435,NC_000962.3,2,1,PR,1,PPE18,+,87.0,30.0,1.0,Q,K,1339435,1339436,lineage4,2


In [236]:
Event_091_PPE18_UnqAASubs_DF = make_unq_aa_subs_df(Event_091_PPE18_SNPs_NSMut_DF)
print(Event_091_PPE18_UnqAASubs_DF.shape)


(1, 4)


In [237]:
Event_091_PPE18_UnqAASubs_DF

,Codon,Ref_AA,Mut_AA,Codon_0based
0,30.0,Q,K,29.0


In [373]:
Event_091_PPE18_AASeq, _ = apply_aa_substitutions(PPE18_AA_Seq, Event_091_PPE18_UnqAASubs_DF)
Event_091_PPE18_AASeq

Mutated position 29: Q → K


Seq('MVDFGALPPEINSARMYAGPGSASLVAAAKMWDSVASDLFSAASAFQSVVWGLT...AAG')

In [374]:
# write_seq_to_fasta(Event_091_PPE18_AASeq, "PPE18_Event091_1Missense", "PPE18.Event091.GCE_AASeq.faa")
